In [ ]:
# INSURANCE CLAIM FRAUD DETECTION
# IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, StandardScaler)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,
                             classification_report,roc_auc_score)
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
import sys

print(sys.executable)
print(sys.version)

# Display all columns
pd.set_option("display.max_columns",None)

#display all rows when needed
pd.set_option("display.max_rows",100)

#for ignoring all the warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# LOAD DATASET
df = pd.read_csv("../data/raw/fraud_oracle.csv")

In [ ]:
# DISPLAYING THE FIRST 5 ROW
df.head()

In [ ]:
# DISPLAYING THE LAST 5 ROW
df.tail()

In [ ]:
# DISPLAYING ALL COLUMNS
df.shape

In [ ]:
# COLUMN NAMES
df.columns

In [ ]:
# TOTAL NUMBER OF COLUMNS
len(df.columns)

In [ ]:
# RANDOM SAMPLES
df.sample(10,random_state=42)

In [ ]:
# DATASET INFORMATION
df.info()

In [ ]:
# STATISTICAL OVERVIEW OF DATASET
df.describe(include='all').T

In [ ]:
# Count unique values in each column and sort from lowest to highest
df.nunique().sort_values()

In [ ]:
# Count how many claims are Fraud (1) and Not Fraud (0)
df["FraudFound_P"].value_counts()

In [ ]:
# Show percentage distribution of target classes
df["FraudFound_P"].value_counts(normalize=True) * 100

In [ ]:
#count missing values in each columns
df.isnull().sum()

In [ ]:
# Count duplicate rows
df.duplicated().sum()

In [ ]:
# FRAUD DISTRIBUTION
plt.figure(figsize=(6,4))
sns.countplot(x="FraudFound_P",data=df)
plt.title("Fraud vs genuine claims")
plt.xlabel("fraud (0 = Genuine, 1 = Fraud)")
plt.ylabel("Number of claims")
plt.show()

In [ ]:
# AGE DISTRIBUTION GRAPH
plt.figure(figsize=(8,5))
sns.histplot(df['Age'],bins=20,kde=True)
plt.title("Distribution of customer age")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()
# CHECK INVALID AGE VALUES
(df["Age"]==0).sum()

In [ ]:
# VEHICLE CATEGORY VS FRAUD
plt.figure(figsize=(8,5))
sns.countplot(x="VehicleCategory",hue="FraudFound_P",data=df)
plt.title("Vehical Categories VS Fraud")
plt.xlabel("vehical category")
plt.ylabel("Number of Claims")
plt.legend(title="Fraud",labels=["Genuine","Fraud"])
plt.show()

In [ ]:
# FINDING THE FRAUD RATE
fraud_rate=(df.groupby("VehicleCategory")["FraudFound_P"].mean()*100)
print(fraud_rate)
(df["Age"]==0).sum()

In [ ]:
# FILTER ROWS WHERE AGE = 0 
# DISPLAY BOTH AGE AND AgeOfPolicyHolder columns
df.loc[df["Age"]==0,["Age","AgeOfPolicyHolder"]].head(20)

In [ ]:
# DROP THE COLUMN WHICH ARE ONLY IDENTIFIER AND DOES NOT AFFECT THE PREDICTIONS
df.drop(columns=["PolicyNumber"],inplace=True)
df.shape 

In [ ]:
# SEPARATE FEATURES (X) AND TARGET (y)
# INPUT FEATURES
X=df.drop(columns=["FraudFound_P"])
print("X Shape :", X.shape)
#TARGET VARIABLE
y=df["FraudFound_P"]
print("y Shape :", y.shape)

In [ ]:
# IDENTIFY CATEGORICAL AND NUMERICAL COLUMNS FOR ENCODING
# GETTING ALL THE CATEGORICAL COLUMNS
categorical_columns=X.select_dtypes(include="object").columns

#GETTING ALL THE NUMERICAL COLUMNS(exclude)
numerical_columns=X.select_dtypes(exclude="object").columns
print("Categorical columns:\n",categorical_columns)
print("\n"+"=="*50+"\n")
print("Numerical columns:\n",numerical_columns)

In [ ]:
# CHECK UNIQUE VALUES OF ORDINAL COLUMNS
ordinal_columns=[
    "VehiclePrice",
    "Days_Policy_Accident",
    "Days_Policy_Claim",
    "PastNumberOfClaims",
    "AgeOfVehicle",
    "AgeOfPolicyHolder",
    "NumberOfSuppliments",
    "AddressChange_Claim",
    "NumberOfCars"
    ]

for col in ordinal_columns:
    print(f"column: {col}")
    print(df[col].unique())
    print(f"{"="*50}")

In [ ]:
# SPLIT FEATURES AND TARGET
from sklearn.model_selection import train_test_split
# Split dataset into training and testing sets
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
# COLUMNS WITH NO NATURAL ORDERS(One-Hot Encoding Columns (Nominal))
onehot_columns=["Month","DayOfWeek","Make","DayOfWeekClaimed","MonthClaimed",
                "MaritalStatus","PolicyType","VehicleCategory","BasePolicy"]

# BINARY COLUMNS(beacuse of only 2 values we can use OrdinalEncoder instead of LabelEncoder)
binary_columns=["Sex","AccidentArea","Fault","PoliceReportFiled","WitnessPresent","AgentType"]
numerical_columns=["WeekOfMonth","WeekOfMonthClaimed","Age","RepNumber","Deductible",
                   "DriverRating","Year"]

# DEFINE ORDER FOR ORDINAL COLUMNS
vehicle_price_order =["less than 20000","20000 to 29000","30000 to 39000","40000 to 59000",
                      "60000 to 69000","more than 69000"]
days_policy_accident_order =["none","1 to 7","8 to 15","15 to 30","more than 30"]
days_policy_claim_order =["none","8 to 15","15 to 30","more than 30"]
past_claims_order =["none","1","2 to 4","more than 4"]
age_vehicle_order =["new","2 years","3 years","4 years","5 years","6 years","7 years","more than 7"]
age_policyholder_order =["16 to 17","18 to 20","21 to 25","26 to 30","31 to 35","36 to 40","41 to 50",
                         "51 to 65","over 65"]
number_suppliments_order =["none","1 to 2","3 to 5","more than 5"]
address_change_order =["no change","under 6 months","1 year","2 to 3 years","4 to 8 years"]
number_cars_order =["1 vehicle","2 vehicles","3 to 4","5 to 8","more than 8"]

In [ ]:
# ORDINAL ENCODER
ordinal_encoder=OrdinalEncoder(categories=[vehicle_price_order, days_policy_accident_order,
                                           days_policy_claim_order, past_claims_order,
                                           age_vehicle_order, age_policyholder_order,
                                           number_suppliments_order, address_change_order,
                                           number_cars_order])

In [ ]:
# PREPROCESSING PIPELINE
preprocessor= ColumnTransformer(
    transformers=[
        # One-Hot Encoding for Nominal Columns
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
            onehot_columns
        ),

         # Ordinal Encoding for Ordered Columns
        (
            "ordinal",
            ordinal_encoder,
            ordinal_columns
        ),

        # Ordinal Encoding for Binary Columns
        (
            "binary",
            OrdinalEncoder(),
            binary_columns
        ),

        # Standard Scaling for Numerical Columns
        (
            "scaler",
            StandardScaler(),
            numerical_columns
        )
    ],
    remainder="drop"
)

In [ ]:
# FIT AND TRANSFORM TRAINING DATA
# model Learns preprocessing rules from training data
X_train_processed = preprocessor.fit_transform(X_train)
# TRANSFORM TEST DATA
# Apply learned preprocessing to testing data
X_test_processed=preprocessor.transform(X_test)

print("Training data shape: ",X_train_processed.shape)
print("Testing data shape: ",X_test_processed.shape)

In [ ]:
# MODEL EVALUATION FUNCTION
def evaluate_model(model, X_train, X_test, y_train, y_test):
    # train the model
    model.fit(X_train,y_train)
    # making the predictions
    y_pred=model.predict(X_test)
    # calculating evalution metrics
    accuracy= accuracy_score(y_test,y_pred)
    precision=precision_score(y_test, y_pred, zero_division=0)
    recall=recall_score(y_test,y_pred, zero_division=0)
    f1=f1_score(y_test,y_pred,zero_division=0)

    # result
    print("="*50)
    print("Model: ", model.__class__.__name__)
    print("="*50)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"f1-score: {f1:.4f}")
    print("Confusion Matrix")
    print(confusion_matrix(y_test, y_pred))

    print("Classification report")
    print(classification_report(y_test,y_pred,zero_division=0))

    return {
        "Model": model.__class__.__name__,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

In [ ]:
# LOGISTIC REGRESSION
logistic_model=LogisticRegression(random_state=42)
lr_result = evaluate_model(logistic_model,X_train_processed,X_test_processed,y_train,y_test)

In [ ]:
# DECISION TREE CLASSIFIER
decision_tree=DecisionTreeClassifier(random_state=42)
dt_result = evaluate_model(decision_tree,X_train_processed,X_test_processed,y_train,y_test)

In [ ]:
# K-NEAREST NEIGHBORS (KNN)
knn_model=KNeighborsClassifier(n_neighbors=5)
knn_result = evaluate_model(knn_model,X_train_processed,X_test_processed,y_train,y_test)

In [ ]:
# MODEL COMPARISON
comparison = pd.DataFrame([
    lr_result,
    dt_result,
    knn_result
])
comparison

In [ ]:
# CREATE SMOTE OBJECT
smote=SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)
# BEFORE SMOTE
print("Before SMOTE")
print(y_train.value_counts())
print("-" * 40)
# AFTER SMOTE
print("After SMOTE")
print(y_train_smote.value_counts())

In [ ]:
# RETRAIN THE MODELS USING BALANCED TARAINING DATA
# LOGISTIC REGRESSION AFTER SMOTE
lr_smote_result=evaluate_model(
    LogisticRegression(random_state=42),
    X_train_smote,
    X_test_processed,
    y_train_smote,
    y_test
)

In [ ]:
# DECISION TREE AFTER SMOTE
dt_smote_result=evaluate_model(
    DecisionTreeClassifier(random_state=42),
    X_train_smote,
    X_test_processed,
    y_train_smote,
    y_test
)

In [ ]:
# KNN AFTER SMOTE
knn_smote_result = evaluate_model(
    KNeighborsClassifier(n_neighbors=5),
    X_train_smote,
    X_test_processed,
    y_train_smote,
    y_test
)

In [ ]:
# PARAMETERS TO TEST
param_grid = {
    "criterion":["gini","entropy"],
    "max_depth":[5,10,15,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

In [ ]:
grid_search = GridSearchCV(
    estimator= DecisionTreeClassifier(random_state=42),
    param_grid= param_grid,
    scoring= "recall",
    cv= 5,
    n_jobs= -1,
    verbose= 2
)
#train gridsearchCV
grid_search.fit(
    X_train_smote,
    y_train_smote
)

In [ ]:
# View the Best Parameters
print("Best Parameters:")
print(grid_search.best_params_)

print()

print("Best Recall:")
print(grid_search.best_score_)

In [ ]:
# best_dt is your tuned Decision Tree.
best_dt=grid_search.best_estimator_
#evaluate it
dt_tuned_result=evaluate_model(
    best_dt,
    X_train_smote,
    X_test_processed,
    y_train_smote,
    y_test
)

In [ ]:
# LOGISTIC REGRESSION PARAMETERS FOR HYPERPARAMETER TUNNING
lr_param_grid={
    "C":[0.001, 0.01, 0.1, 1, 10, 100],
    "solver":["linlinear","lbfgs"],
    "penalty":["l2"]
}

In [ ]:
# LOGISTIC REGRESSION GRID SEARCH
lr_grid=GridSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_grid=lr_param_grid,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=2
)
#Train
lr_grid.fit(
    X_train_smote,
    y_train_smote
)

In [ ]:
# Finding Best Parameters
print("Best Parameters: ")
print(lr_grid.best_params_)
print()
print("Best recall:")
print(lr_grid.best_score_)

In [ ]:
# best_lr is your tunned logistic regression
best_lr=lr_grid.best_estimator_
lr_tuned_result=evaluate_model(
    best_lr,
    X_train_smote,
    X_test_processed,
    y_train_smote,
    y_test
)

In [ ]:
# KNN HYPERPARAMETERS
knn_param_grid={
    "n_neighbors":[3,5,7,9,11,15,21],
    "weights":["uniform","distance"],
    "metric":["euclidean","manhattan"]
}

In [ ]:
# KNN GRIDSEARCH
knn_grid=GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=knn_param_grid,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=2
)

#TRAIN KNN SEARCH
knn_grid.fit(X_train_smote,y_train_smote)

In [ ]:
# FINDING THE BEST PARAMETERS
print("Best Parameters: ")
print(knn_grid.best_params_)
print("\n Best Recall: ")
print(knn_grid.best_score_)

In [ ]:
# Get the best KNN
best_knn = knn_grid.best_estimator_
knn_tuned_result=evaluate_model(
    best_knn,
    X_train_smote,
    X_test_processed,
    y_train_smote,
    y_test
)

In [ ]:
# FINAL MODEL COMPARISON
final_results=pd.DataFrame({
    "Model":[
        "Logistic regression",
        "Decision Tree",
        "KNN"
    ],
    "Accuracy":[
        0.6352,
        0.7010,
        0.6644
    ],
    "Precision":[
        0.1311,
        0.1377,
        0.1072
    ],
    "Recall":[
        0.9027,
        0.7568,
        0.6270
    ],
    "F1-Score":[
        0.2289,
        0.2329,
        0.1831
    ]
})
final_results


In [ ]:
print("Selected Model : Logistic Regression")
## Why Logistic Regression?

#- Business objective is fraud detection.
#- Highest Recall (90.27%).
#- Detected 167 out of 185 fraud claims.
#- Missed only 18 fraud cases.
#- Therefore Logistic Regression is selected as the final model.

In [ ]:
import joblib
joblib.dump(best_lr,"../models/fraud_model.pkl")

In [ ]:
joblib.dump(preprocessor, "../models/preprocessor.pkl")

In [ ]:
import os
print(os.listdir("../models"))

In [63]:
# Get one fraud record
fraud_record = df[df["FraudFound_P"] == 1].sample(1, random_state=42)

# Separate input and output
X_test = fraud_record.drop("FraudFound_P", axis=1)

# Transform using saved preprocessor
X_processed = preprocessor.transform(X_test)

# Predict
prediction = best_lr.predict(X_processed)

# Probability
probability = best_lr.predict_proba(X_processed)

print("Prediction:", prediction)
print("Probability:", probability)

# Show all values of this record
fraud_record.T

Prediction: [1]
Probability: [[0.43790619 0.56209381]]


,5079
Month,Jul
WeekOfMonth,3
DayOfWeek,Thursday
Make,Toyota
AccidentArea,Rural
DayOfWeekClaimed,Monday
MonthClaimed,Jul
WeekOfMonthClaimed,3
Sex,Male
MaritalStatus,Married
